In [ ]:
import pandas as pd
import numpy as np
#
# from sqlalchemy import create_engine, MetaData, Table
from matplotlib import font_manager, rc
import platform
import requests
import matplotlib.pyplot as plt
import statsmodels.api as sm
from binance.um_futures import UMFutures
from scipy.stats import t, normaltest
from datetime import datetime, timedelta
from joblib import Parallel, delayed, parallel_backend
import numba
from tqdm import tqdm
from datetime import datetime
import threading
import warnings
from itertools import combinations
from numpy.linalg import inv
from pathlib import Path
import time
from config import *

warnings.filterwarnings("ignore")

# 운영체제에 따라 적절한 한글 폰트 설정
if platform.system() == 'Darwin':  # macOS의 경우
    rc('font', family='AppleGothic')

plt.rcParams['axes.unicode_minus'] = False

# 모든 행과 열을 출력하도록 옵션 설정
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

# 혹은 출력 너비를 늘리기 위해서도 설정할 수 있습니다.
pd.set_option('display.width', 1000)

In [ ]:
import json
from binance.client import Client

client = Client(API_KEY, SECRET_KEY)

In [ ]:
START_DATE = "2023-06-01"
END_DATE = "2025-12-01"
start_ts = int(pd.to_datetime(START_DATE).timestamp() * 1000)
end_ts = int(pd.to_datetime(END_DATE).timestamp() * 1000)
klines = client.futures_klines(
                symbol='BTCUSDT',
                interval='1h',
                startTime=start_ts,
                endTime=end_ts,
                limit=10
            )

klines

## PV 데이터 종목별 폴더 저장

In [ ]:
import pandas as pd
import numpy as np
from binance.client import Client
from tqdm import tqdm
from datetime import datetime
import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import threading

# =============================================
# 0️⃣ 설정
# =============================================

BASE_DIR = Path("/Users/cyberedjs/Desktop/Unity/data/symbols")

START_DATE = "2023-06-01"
END_DATE = "2025-12-01"
INTERVAL = "4h"
MAX_WORKERS = 3

# =============================================
# 1️⃣ 심볼 리스트 로드
# =============================================
def get_usdt_symbols():
    info = client.futures_exchange_info()
    symbols = [
        s["symbol"]
        for s in info["symbols"]
        if s["symbol"].endswith("USDT")
    ]
    exclude = ["USDCUSDT", "USTCUSDT", "BUSDUSDT", "FDUSDUSDT"]
    return [s for s in symbols if s not in exclude]

# =============================================
# 2️⃣ 개별 심볼 데이터 수집 및 저장 함수 (핵심 변경)
# =============================================
def fetch_and_save_symbol(symbol, start_ts, end_ts):
    """
    개별 심볼의 전체 기간 데이터를 수집하여 즉시 Parquet 파일로 저장
    """
    symbol_short = symbol.replace("USDT", "")
    symbol_dir = BASE_DIR / symbol_short
    symbol_dir.mkdir(parents=True, exist_ok=True)

    save_path = symbol_dir / f"ohlcv_{INTERVAL}.parquet"

    all_klines = []
    current_start = start_ts
    LIMIT = 1500  # 바이낸스 최대 요청 개수

    try:
        while True:
            # 데이터 요청
            klines = client.futures_klines(
                symbol=symbol,
                interval=INTERVAL,
                startTime=current_start,
                endTime=end_ts,
                limit=LIMIT
            )
            
            if not klines:
                break
            
            all_klines.extend(klines)
            
            # 다음 요청 시작 시간 갱신 (마지막 캔들 OpenTime + 1ms)
            last_open_time = klines[-1][0]
            current_start = last_open_time + 1
            
            # 종료 조건
            if current_start >= end_ts or len(klines) < LIMIT:
                break
            
            # 너무 빠른 요청 방지
            time.sleep(0.8)

        if not all_klines:
            return f"{symbol} (No Data)"

        # DataFrame 생성
        df = pd.DataFrame(all_klines, columns=[
            "timestamp", "open", "high", "low", "close", "volume",
            "close_time", "quote_volume", "trades", "taker_buy_base", "taker_buy_quote", "ignore"
        ])

        # 전처리
        df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
        df.set_index("timestamp", inplace=True)
        
        # 필요한 컬럼만 숫자형 변환 및 선택 (OHLCV)
        cols = ["open", "high", "low", "close", "volume"]
        for col in cols:
            df[col] = pd.to_numeric(df[col], errors="coerce")
        
        df = df[cols] # 최종 컬럼 필터링
        
        # 중복 제거 및 정렬
        df = df[~df.index.duplicated(keep="last")].sort_index()

        # 🔥 [핵심] 개별 파일로 즉시 저장
        df.to_parquet(save_path, engine="fastparquet", compression="snappy")
        
        return f"{symbol} ({len(df)} rows)"

    except Exception as e:
        return f"❌ {symbol} Error: {str(e)}"

# =============================================
# 3️⃣ 메인 실행 루프
# =============================================
def download_all_data():
    if client is None:
        print("❌ Binance Client가 설정되지 않았습니다.")
        return

    # 시간 변환
    start_ts = int(pd.to_datetime(START_DATE).timestamp() * 1000)
    end_ts = int(pd.to_datetime(END_DATE).timestamp() * 1000)

    all_symbols = get_usdt_symbols()

    # 2. 이미 파일이 존재하는지 확인하여 대상 필터링
    symbols_to_download = []
    for sym in all_symbols:
        symbol_short = sym.replace("USDT", "")
        save_path = BASE_DIR / symbol_short / f"ohlcv_{INTERVAL}.parquet"
        
        # 파일이 이미 존재하면 리스트에 추가하지 않음
        if save_path.exists():
            continue
        
        symbols_to_download.append(sym)

    skipped_count = len(all_symbols) - len(symbols_to_download)
    print(f"⏭ 이미 존재하는 {skipped_count}개 종목을 제외합니다.")
    print(f"📊 총 {len(symbols_to_download)}개 종목 다운로드 시작 -> {BASE_DIR}")
    print(symbols_to_download)

    if not symbols_to_download:
        print("✅ 모든 종목이 이미 다운로드되어 있습니다.")
        return

    results = []
    
    # 스레드 풀 실행 (필터링된 리스트 사용)
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {
            executor.submit(fetch_and_save_symbol, sym, start_ts, end_ts): sym 
            for sym in symbols_to_download
        }

        with tqdm(total=len(symbols_to_download), desc="Downloading Symbols") as pbar:
            for future in as_completed(futures):
                res = future.result()
                results.append(res)
                pbar.update(1)
                time.sleep(2)
 
    print("\n✅ 누락된 데이터 수집 완료.")

if __name__ == "__main__":
    download_all_data()

## 데이터 타입별로 모든 종목 데이터 합치기

In [ ]:
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import os

# =============================================
# 0️⃣ 설정
# =============================================

# 데이터를 읽어올 루트 경로 (기존 다운로드 경로와 동일)
BASE_DIR = Path("/Users/cyberedjs/Desktop/Unity/data/symbols")
COMBINED_DIR = Path("/Users/cyberedjs/Desktop/Unity/data/combined")

# 데이터를 저장할 새로운 'all' 디렉토리
ALL_DIR = BASE_DIR / "combined"
ALL_DIR.mkdir(parents=True, exist_ok=True)

# 병합할 데이터 타입
DATA_TYPES = ["open", "close", "high", "low", "volume"]
INTERVAL = '1h'

# =============================================
# 1️⃣ 데이터 타입별 병합 및 저장 함수
# =============================================
def combine_data_by_type(data_type: str):
    """
    특정 데이터 타입(open, close 등)에 대해 모든 종목의 데이터를 병합하고 저장합니다.
    
    Args:
        data_type (str): 병합할 OHLCV 컬럼 이름 ("open", "close", "high", "low", "volume").
    """
    
    print(f"\n🔄 {data_type.upper()} 데이터 병합 시작...")
    
    # 1. 모든 종목 폴더 찾기
    # BASE_DIR 아래의 모든 디렉토리를 찾고, 그 안에 ohlcv_15m.parquet 파일이 있는지 확인
    symbol_dirs = [p.parent for p in BASE_DIR.glob(f"*/ohlcv_{INTERVAL}.parquet")]

    if not symbol_dirs:
        print("❌ Parquet 파일이 있는 종목 폴더를 찾을 수 없습니다.")
        return

    # 병합할 데이터프레임 조각들을 담을 리스트
    data_frames = []

    for symbol_dir in tqdm(symbol_dirs, desc=f"Loading {data_type} data"):
        try:
            # 폴더 이름으로부터 심볼 이름 추출 (예: /data/BTC -> BTC)
            symbol_short = symbol_dir.name 
            file_path = symbol_dir / f"ohlcv_{INTERVAL}.parquet"
            
            # 2. 파일 읽기
            # 필요한 컬럼(인덱스와 data_type 컬럼)만 효율적으로 읽어옵니다.
            df_symbol = pd.read_parquet(file_path, columns=[data_type])

            # 3. 컬럼 이름 변경 (종목 심볼로 변경)
            df_symbol.rename(columns={data_type: symbol_short}, inplace=True)
            
            data_frames.append(df_symbol)

        except Exception as e:
            print(f"⚠️ {symbol_short} 파일 처리 중 오류 발생: {e}")
            continue

    if not data_frames:
        print(f"❌ 병합할 {data_type} 데이터가 없습니다.")
        return

    # 4. 모든 데이터프레임 병합 (Outer Join, 인덱스 기준)
    # pd.concat을 사용하면 서로 다른 기간을 가진 데이터가 자동으로 NaN으로 채워지며 병합됩니다.
    combined_df = pd.concat(data_frames, axis=1)
    
    # 5. 시간 순으로 정렬
    combined_df = combined_df.sort_index()

    # 6. 저장
    output_filename = f"{data_type}_{INTERVAL}.parquet"
    output_path = COMBINED_DIR / output_filename
    
    combined_df.to_parquet(output_path, engine="fastparquet", compression="snappy")

    print(f"✅ {data_type.upper()} 데이터 저장 완료: {output_path}")
    print(f" -> Shape: {combined_df.shape}")


# =============================================
# 2️⃣ 실행
# =============================================
if __name__ == "__main__":
    
    # 각 데이터 타입에 대해 함수 실행
    for data_type in DATA_TYPES:
        combine_data_by_type(data_type)

    print("\n🎉 모든 OHLCV 데이터 타입 병합 및 저장 완료. (BASE_DIR/all 디렉토리 확인)")

In [25]:
close = pd.read_parquet(f"{COMBINED_DIR}/close_4h.parquet", engine="pyarrow")
close.tail()

,CELO,MKR,EDU,LUNA2,XPIN,TOKEN,HMSTR,VIC,FOLKS,DOGS,STABLE,LOKA,SSV,KDA,PROM,MOODENG,ONG,IMX,PONKE,SIREN,AEVO,ADA,AKE,SAFE,ORBS,VET,ORDER,TREE,SOLV,REI,MOVE,INIT,ALPINE,REN,CHR,GALA,STBL,BTC,KAIA,YB,HOME,TROY,SPK,DEGO,ZETA,BRETT,TLM,ZEC,EDEN,IDEX,REZ,TIA,KGEN,FARTCOIN,SC,FTM,LTC,SPX,PLUME,ENA,1000PEPE,SUN,MEMEFI,VELVET,OMG,ALCH,BARD,SUI,POL,LINA,AGT,INJ,JTO,1000LUNC,1000SATS,IOST,DENT,IDOL,FORM,NULS,GHST,OCEAN,ENS,CHZ,GAS,RESOLV,USELESS,RIVER,CELR,ONT,GMX,AUCTION,CYBER,FXS,ZK,HIGH,OGN,APE,BIO,PORTAL,1000000MOG,HAEDAL,CTSI,SKY,XRP,LUMIA,COTI,HUMA,B2,CVX,KSM,BLUAI,BEAMX,PENDLE,MINA,FLM,SKL,CARV,1000RATS,UAI,MORPHO,MEME,NOT,KAITO,ZRO,YGG,SNT,TRU,BCH,BB,NEO,BICO,TURTLE,DOT,CVC,B3,DF,BEL,UXLINK,AI16Z,HIPPO,PTB,C98,GPS,KLAY,FIS,LISTA,MAGIC,WAL,FLUX,BOB,ICNT,XTZ,FIO,CATI,AVA,NIL,ATOM,FHE,BROCCOLI714,WLD,VANA,A2Z,BROCCOLIF3B,NKN,VINE,DAR,RVV,EVAA,OL,DGB,AWE,BANK,AERGO,USUAL,GRASS,TURBO,SHELL,SLP,POWR,TOSHI,BAL,SFP,OP,PENGU,AGIX,MILK,BOME,AXS,IN,OBOL,SPELL,AIXBT,RSR,BNT,BOND,PIEVERSE,BDXN,OXT,AVAAI,ENSO,PUMPBTC,LDO,STEEM,BEAT,SCR,OM,CRV,CLANKER,BAND,ETHW,ERA,HBAR,METIS,PERP,WCT,RAYSOL,LEVER,LAB,MANTA,WIF,BANANAS31,KAVA,IO,AIOT,ZEREBRO,FIDA,DMC,TUT,SOL,VVV,BMT,PROVE,ETC,TAC,NTRN,CROSS,SXT,RUNE,CLO,ALPACA,XNY,STX,SKYAI,RONIN,1000SHIB,ALGO,1000000BOB,BSV,KOMA,ACH,RENDER,MIRA,ARPA,ALL,1MBABYDOGE,LSK,FUN,AIO,42,JUP,THE,ANKR,QUICK,G,ICP,MET,AIA,ACT,DEEP,VOXEL,MLN,LQTY,NAORIS,QTUM,ASTER,AERO,ROSE,ZKJ,HOLO,KEY,IOTA,SENT,JST,BSW,2Z,GOAT,COMP,T,PHB,S,PNUT,ZKC,JELLYJELLY,EIGEN,SOMI,COS,SLERF,A,XAN,COAI,CHILLGUY,XAI,PORT3,DODOX,F,VELODROME,FLUID,MERL,ICX,ALICE,MON,SAGA,H,AIN,PYTH,DOOD,1000XEC,DRIFT,MBOX,ZEN,MMT,RIF,1000X,AKT,UNFI,POPCAT,RECALL,GMT,RDNT,DOGE,XEM,LRC,FTT,RED,CUDIS,MAV,ZIL,SKATE,KAS,BULLA,CKB,HYPE,0G,DUSK,PIXEL,ORDI,HANA,ENJ,BTR,TRADOOR,SYN,COMMON,KNC,HEMI,AMB,UB,SYS,MASK,PUNDIX,BERA,VFY,TON,AAVE,ETHFI,GRIFFAIN,NEIROETH,LAYER,RLC,XLM,MOVR,NEWT,STMX,RARE,SOON,ONE,XCN,LINK,NXPC,NEIRO,MDT,PROMPT,ZORA,ALLO,SUPER,TAKE,PUFFER,ARIA,EPIC,UMA,QNT,UNI,ASR,TRX,SCRT,BLESS,AI,APT,BLUR,1INCH,HFT,ME,ONDO,PARTI,HEI,BLZ,VIRTUAL,BIGTIME,JCT,VTHO,MANA,AR,PEOPLE,POLYX,TRUTH,WAVES,PUMP,GUN,FET,APR,GLMR,REEF,JOE,ZRC,币安人生,STPT,TAIKO,SUSHI,SNX,MYX,FIL,DIA,SWELL,HOT,WAXP,AT,CHESS,HIVE,API3,EPT,BID,ZRX,KITE,VANRY,TRB,DASH,EUL,TANSSI,TWT,DYDX,NFP,PIPPIN,NOM,BR,RVN,NEAR,TRUST,ARB,1000FLOKI,DAM,XVG,LIGHT,SWARMS,COOKIE,BAKE,ETH,AVAX,BAT,AGLD,MELANIA,MAVIA,YALA,JASMY,CGPT,FLOCK,BAS,ARK,MYRO,ID,MTL,VIDT,WOO,RPL,BNB,BANANA,CC,AXL,TST,ASTR,XPL,SEI,DEXE,TOWNS,GTC,BTCDOM,STRK,BABY,SYRUP,FF,THETA,1000WHY,GRT,ANIME,AVNT,BNX,EGLD,MUBARAK,ARC,SAHARA,PLAY,OG,YFI,XVS,ATH,ARKM,BAN,PAXG,STRAX,SANTOS,NMR,IP,CTK,WLFI,SOPH,LA,ATA,ON,MOCA,M,GLM,ALPHA,SONIC,IRYS,COMBO,STG,FORTH,TNSR,IOTX,C,SIGN,ALT,D,MEW,1000CHEEMS,LINEA,XMR,HYPER,TAG,TRUMP,DEFI,SXP,STORJ,TA,HOOK,ACE,Q,RAY,4,LOOM,SQD,SAPIEN,COW,OPEN,RAD,ILV,STO,B,KERNEL,CAKE,LPT,MITO,DYM,DOLO,LYN,ACX,ORCA,HIFI,TAO,GIGGLE,ESPORTS,BADGER,ZBT,CETUS,OMNI,PHA,KMNO,1000CAT,1000BONK,SAND,DEGEN,CFX,FLOW,W
timestamp,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2025-11-30 08:00:00,0.169,1650.1,0.1708,0.07226,0.002116,0.004654,0.000237,0.1075,9.521,0.000046,0.02997,0.11323,3.593,0.02476,8.970,0.07671,0.09912,0.3016,0.03807,0.08438,0.04494,0.4174,0.000424,0.1545,0.035157,0.013304,0.10924,0.1322,0.01918,0.00892,0.04999,0.1205,0.5928,0.048,0.0545,0.00759,0.06229,91000.9,0.07884,0.4881,0.023404,0.000187,0.02928,0.5931,0.0894,0.01726,0.002443,448.74,0.07463,0.0561,0.006464,0.6628,0.20365,0.3170,0.003653,0.7702